In [3]:
import os
import re
import time
import json
import base64
import secrets
from pathlib import Path
from datetime import datetime, timezone

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from web3 import Web3
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.hkdf import HKDF
from cryptography.hazmat.primitives.ciphers.aead import AESGCM


# ============================================================
# 1. Configuration
# ============================================================

load_dotenv()

KEYS_DIR = Path("keys")
RESULTS_DIR = Path("results")
FIGURES_DIR = Path("figures")

RESULTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

STATION_ID = "ST01"
STATION_SECRET_PATH = KEYS_DIR / f"{STATION_ID}_secret.pem"

PAYLOAD_SIZES = {
    "1 KB": 1 * 1024,
    "100 KB": 100 * 1024,
    "1 MB": 1 * 1024 * 1024,
    "5 MB": 5 * 1024 * 1024,
}

RUNS_PER_SIZE = 5
SLEEP_BETWEEN_RUNS = 2.0

# Set these to False if you want a local benchmark only.
ENABLE_IPFS = True
ENABLE_BLOCKCHAIN = True

# Pinata
PINATA_API_KEY="b47ab0b55da5bd156612"
PINATA_SECRET_API_KEY="4ae8fffb555559daa06be3cf6d3ec7c90bb6029867f947af84c3f93f9be41746"
PINATA_URL = "https://api.pinata.cloud/pinning/pinJSONToIPFS"

# Blockchain
PRIVATE_KEY = os.getenv("PRIVATE_KEY", "").strip().replace('"', "")
IOTA_RPC_URL = os.getenv("IOTA_RPC_URL", "https://json-rpc.evm.testnet.iota.cafe").strip().replace('"', "")
IOTA_CONTRACT_ADDRESS = os.getenv("IOTA_CONTRACT_ADDRESS", "").strip().replace('"', "")

# Cost estimation
IOTA_USD = float(os.getenv("IOTA_USD", "0.06143"))

# Minimal ABI
MILEAGE_LEDGER_ABI = [
    {
        "inputs": [
            {"internalType": "string", "name": "vehicleId", "type": "string"},
            {"internalType": "string", "name": "timestamp", "type": "string"},
            {"internalType": "string", "name": "proofCid", "type": "string"},
            {"internalType": "uint256", "name": "odometerKm", "type": "uint256"},
        ],
        "name": "recordMileage",
        "outputs": [],
        "stateMutability": "nonpayable",
        "type": "function",
    }
]


# ============================================================
# 2. Style for Q1-like figures
# ============================================================

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 10,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.linewidth": 0.8,
    "figure.dpi": 300,
    "savefig.dpi": 600,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

COLORS = {
    "Acquisition": "#BFD7E5",
    "Crypto": "#164E7A",
    "IPFS": "#6F8FA3",
    "Blockchain": "#D9E6EF",
}


# ============================================================
# 3. Helpers
# ============================================================

def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def load_station_secret_pem(path: Path) -> bytes:
    if not path.exists():
        raise FileNotFoundError(f"Station secret not found: {path}")

    text = path.read_text(encoding="utf-8")
    b64 = re.sub(
        r"-----BEGIN STATION SHARED SECRET-----|-----END STATION SHARED SECRET-----|\s+",
        "",
        text,
    )
    return base64.b64decode(b64)


def derive_aes_key(shared_secret: bytes, salt: bytes) -> bytes:
    hkdf = HKDF(
        algorithm=hashes.SHA256(),
        length=32,
        salt=salt,
        info=b"vehicle-evidence-aes-gcm-key",
    )
    return hkdf.derive(shared_secret)


def encrypt_payload(shared_secret: bytes, payload: bytes, aad: bytes) -> dict:
    salt = secrets.token_bytes(16)
    nonce = secrets.token_bytes(12)

    key = derive_aes_key(shared_secret, salt)
    aesgcm = AESGCM(key)

    ciphertext = aesgcm.encrypt(nonce, payload, aad)

    return {
        "salt_b64": base64.b64encode(salt).decode("ascii"),
        "nonce_b64": base64.b64encode(nonce).decode("ascii"),
        "aad_b64": base64.b64encode(aad).decode("ascii"),
        "ciphertext_b64": base64.b64encode(ciphertext).decode("ascii"),
        "ciphertext_size_bytes": len(ciphertext),
    }


def upload_to_pinata(bundle: dict, station_id: str, payload_label: str, run_id: int) -> str:
    if not ENABLE_IPFS:
        return f"simulated-cid-{station_id}-{payload_label}-{run_id}"

    if not PINATA_API_KEY or not PINATA_SECRET_API_KEY:
        raise RuntimeError("PINATA_API_KEY or PINATA_SECRET_API_KEY missing in .env")

    headers = {
        "pinata_api_key": PINATA_API_KEY,
        "pinata_secret_api_key": PINATA_SECRET_API_KEY,
        "Content-Type": "application/json",
    }

    pinata_payload = {
        "pinataMetadata": {
            "name": f"evidence_{station_id}_{payload_label}_{run_id}_{int(time.time())}"
        },
        "pinataContent": bundle,
    }

    last_error = None

    for attempt in range(1, 4):
        try:
            response = requests.post(
                PINATA_URL,
                headers=headers,
                json=pinata_payload,
                timeout=180,
            )
            response.raise_for_status()
            return response.json()["IpfsHash"]
        except Exception as e:
            last_error = e
            time.sleep(2 * attempt)

    raise RuntimeError(f"Pinata upload failed after retries: {last_error}")


def init_blockchain():
    if not ENABLE_BLOCKCHAIN:
        return None, None, None

    if not PRIVATE_KEY or not IOTA_CONTRACT_ADDRESS:
        raise RuntimeError("PRIVATE_KEY or IOTA_CONTRACT_ADDRESS missing in .env")

    w3 = Web3(Web3.HTTPProvider(IOTA_RPC_URL, request_kwargs={"timeout": 120}))

    if not w3.is_connected():
        raise RuntimeError("Cannot connect to IOTA RPC")

    account = w3.eth.account.from_key(PRIVATE_KEY)

    contract = w3.eth.contract(
        address=Web3.to_checksum_address(IOTA_CONTRACT_ADDRESS),
        abi=MILEAGE_LEDGER_ABI,
    )

    code = w3.eth.get_code(Web3.to_checksum_address(IOTA_CONTRACT_ADDRESS))
    if code == b"":
        raise RuntimeError("No contract code found at IOTA_CONTRACT_ADDRESS")

    return w3, account, contract


def build_tx_params(w3, account_address, function_call):
    nonce = w3.eth.get_transaction_count(account_address, "pending")

    try:
        gas_estimate = function_call.estimate_gas({"from": account_address})
        gas_limit = int(gas_estimate * 1.25)
    except Exception:
        gas_limit = 500000

    latest_block = w3.eth.get_block("latest")
    base_fee = latest_block.get("baseFeePerGas", None)

    params = {
        "from": account_address,
        "nonce": nonce,
        "gas": gas_limit,
    }

    if base_fee is not None:
        priority_fee = w3.to_wei(1, "gwei")
        max_fee = int(base_fee * 2 + priority_fee)
        params["maxFeePerGas"] = max_fee
        params["maxPriorityFeePerGas"] = priority_fee
    else:
        params["gasPrice"] = int(w3.eth.gas_price * 1.15)

    return params


def send_blockchain_tx(w3, account, contract, vehicle_id, timestamp, cid, odometer_km):
    if not ENABLE_BLOCKCHAIN:
        return {
            "tx_hash": "simulated-tx",
            "gas_used": None,
            "effective_gas_price_wei": None,
            "cost_native": None,
            "cost_usd": None,
            "success": True,
        }

    function_call = contract.functions.recordMileage(
        vehicle_id,
        timestamp,
        cid,
        int(odometer_km),
    )

    last_error = None

    for attempt in range(1, 4):
        try:
            tx_params = build_tx_params(w3, account.address, function_call)
            tx = function_call.build_transaction(tx_params)
            signed = account.sign_transaction(tx)

            raw_tx = getattr(signed, "raw_transaction", None)
            if raw_tx is None:
                raw_tx = getattr(signed, "rawTransaction")

            tx_hash = w3.eth.send_raw_transaction(raw_tx)
            receipt = w3.eth.wait_for_transaction_receipt(tx_hash, timeout=300)

            gas_used = int(receipt["gasUsed"])

            effective_gas_price = receipt.get("effectiveGasPrice", None)
            if effective_gas_price is None:
                effective_gas_price = tx.get("gasPrice", w3.eth.gas_price)

            effective_gas_price = int(effective_gas_price)

            cost_wei = gas_used * effective_gas_price
            cost_native = float(w3.from_wei(cost_wei, "ether"))
            cost_usd = cost_native * IOTA_USD

            return {
                "tx_hash": tx_hash.hex(),
                "gas_used": gas_used,
                "effective_gas_price_wei": effective_gas_price,
                "cost_native": cost_native,
                "cost_usd": cost_usd,
                "success": bool(receipt["status"] == 1),
            }

        except Exception as e:
            last_error = e
            time.sleep(3 * attempt)

    raise RuntimeError(f"Blockchain transaction failed after retries: {last_error}")


def simulate_obd_read():
    # Replace this with real OBD read if needed.
    time.sleep(0.01)
    return True


def build_payload(size_bytes: int) -> bytes:
    return secrets.token_bytes(size_bytes)


def make_aad(vehicle_id: str, timestamp: str, odometer_km: int, station_id: str) -> bytes:
    return f"{vehicle_id}|{timestamp}|{odometer_km}|{station_id}".encode("utf-8")


# ============================================================
# 4. Benchmark
# ============================================================

def run_benchmark():
    shared_secret = load_station_secret_pem(STATION_SECRET_PATH)
    w3, account, contract = init_blockchain()

    rows = []

    for payload_label, payload_size in PAYLOAD_SIZES.items():
        for run in range(1, RUNS_PER_SIZE + 1):
            vehicle_id = f"V-{STATION_ID}-{payload_label.replace(' ', '')}-{run}-{int(time.time())}"
            timestamp = now_iso()
            odometer_km = 1000 + run

            row = {
                "station_id": STATION_ID,
                "payload_label": payload_label,
                "payload_size_bytes": payload_size,
                "run": run,
                "vehicle_id": vehicle_id,
                "odometer_km": odometer_km,
                "timestamp": timestamp,
                "acquisition_ms": None,
                "crypto_ms": None,
                "ipfs_s": None,
                "blockchain_s": None,
                "total_s": None,
                "cid": None,
                "tx_hash": None,
                "gas_used": None,
                "cost_native": None,
                "cost_usd": None,
                "success": False,
                "error": "",
            }

            t_total_start = time.perf_counter()

            try:
                # 1. Acquisition
                t0 = time.perf_counter()
                simulate_obd_read()
                t1 = time.perf_counter()

                # 2. Payload + Crypto
                payload = build_payload(payload_size)
                aad = make_aad(vehicle_id, timestamp, odometer_km, STATION_ID)

                t2_start = time.perf_counter()
                encrypted_bundle = encrypt_payload(shared_secret, payload, aad)
                t2_end = time.perf_counter()

                bundle_for_ipfs = {
                    "station_id": STATION_ID,
                    "vehicle_id": vehicle_id,
                    "timestamp": timestamp,
                    "odometer_km": odometer_km,
                    "payload_label": payload_label,
                    "crypto": "Kyber-derived HKDF + AES-GCM-256",
                    "bundle": encrypted_bundle,
                }

                # 3. IPFS
                t3_start = time.perf_counter()
                cid = upload_to_pinata(bundle_for_ipfs, STATION_ID, payload_label, run)
                t3_end = time.perf_counter()

                # 4. Blockchain
                t4_start = time.perf_counter()
                tx_result = send_blockchain_tx(
                    w3=w3,
                    account=account,
                    contract=contract,
                    vehicle_id=vehicle_id,
                    timestamp=timestamp,
                    cid=cid,
                    odometer_km=odometer_km,
                )
                t4_end = time.perf_counter()

                t_total_end = time.perf_counter()

                row.update({
                    "acquisition_ms": (t1 - t0) * 1000,
                    "crypto_ms": (t2_end - t2_start) * 1000,
                    "ipfs_s": (t3_end - t3_start),
                    "blockchain_s": (t4_end - t4_start),
                    "total_s": (t_total_end - t_total_start),
                    "cid": cid,
                    "tx_hash": tx_result["tx_hash"],
                    "gas_used": tx_result["gas_used"],
                    "cost_native": tx_result["cost_native"],
                    "cost_usd": tx_result["cost_usd"],
                    "success": tx_result["success"],
                })

                print(
                    f"[OK] {payload_label} run={run} | "
                    f"crypto={row['crypto_ms']:.2f} ms | "
                    f"IPFS={row['ipfs_s']:.2f} s | "
                    f"chain={row['blockchain_s']:.2f} s | "
                    f"total={row['total_s']:.2f} s"
                )

            except Exception as e:
                row["error"] = str(e)[:500]
                row["total_s"] = time.perf_counter() - t_total_start
                print(f"[FAIL] {payload_label} run={run}: {row['error']}")

            rows.append(row)
            time.sleep(SLEEP_BETWEEN_RUNS)

    raw_df = pd.DataFrame(rows)
    raw_path = RESULTS_DIR / "integration_overhead_raw.csv"
    raw_df.to_csv(raw_path, index=False)

    success_df = raw_df[raw_df["success"] == True].copy()

    summary_df = (
        success_df.groupby(["payload_label", "payload_size_bytes"], as_index=False)
        .agg(
            runs=("run", "count"),
            acquisition_ms_mean=("acquisition_ms", "mean"),
            acquisition_ms_std=("acquisition_ms", "std"),
            crypto_ms_mean=("crypto_ms", "mean"),
            crypto_ms_std=("crypto_ms", "std"),
            ipfs_s_mean=("ipfs_s", "mean"),
            ipfs_s_std=("ipfs_s", "std"),
            blockchain_s_mean=("blockchain_s", "mean"),
            blockchain_s_std=("blockchain_s", "std"),
            total_s_mean=("total_s", "mean"),
            total_s_std=("total_s", "std"),
            gas_used_mean=("gas_used", "mean"),
            cost_usd_mean=("cost_usd", "mean"),
        )
    )

    total_runs = raw_df.groupby("payload_label")["run"].count().rename("total_runs")
    success_runs = success_df.groupby("payload_label")["run"].count().rename("successful_runs")
    rates = pd.concat([total_runs, success_runs], axis=1).fillna(0)
    rates["success_rate_percent"] = (rates["successful_runs"] / rates["total_runs"]) * 100

    summary_df = summary_df.merge(
        rates[["success_rate_percent"]],
        left_on="payload_label",
        right_index=True,
        how="left",
    )

    order_map = {label: i for i, label in enumerate(PAYLOAD_SIZES.keys())}
    summary_df["order"] = summary_df["payload_label"].map(order_map)
    summary_df = summary_df.sort_values("order").drop(columns=["order"])

    summary_path = RESULTS_DIR / "integration_overhead_summary.csv"
    summary_df.to_csv(summary_path, index=False)

    print("\nSaved results:")
    print(raw_path)
    print(summary_path)

    return raw_df, summary_df


# ============================================================
# 5. Figures
# ============================================================

def plot_stacked_latency(summary_df: pd.DataFrame):
    labels = summary_df["payload_label"].tolist()

    acquisition_s = summary_df["acquisition_ms_mean"].to_numpy() / 1000
    crypto_s = summary_df["crypto_ms_mean"].to_numpy() / 1000
    ipfs_s = summary_df["ipfs_s_mean"].to_numpy()
    blockchain_s = summary_df["blockchain_s_mean"].to_numpy()

    x = np.arange(len(labels))

    fig, ax = plt.subplots(figsize=(7.2, 4.2))

    ax.bar(x, acquisition_s, label="Acquisition", color=COLORS["Acquisition"], edgecolor="black", linewidth=0.4)
    ax.bar(x, crypto_s, bottom=acquisition_s, label="Crypto", color=COLORS["Crypto"], edgecolor="black", linewidth=0.4)
    ax.bar(x, ipfs_s, bottom=acquisition_s + crypto_s, label="IPFS", color=COLORS["IPFS"], edgecolor="black", linewidth=0.4)
    ax.bar(
        x,
        blockchain_s,
        bottom=acquisition_s + crypto_s + ipfs_s,
        label="Blockchain",
        color=COLORS["Blockchain"],
        edgecolor="black",
        linewidth=0.4,
    )

    totals = acquisition_s + crypto_s + ipfs_s + blockchain_s
    for i, total in enumerate(totals):
        ax.annotate(
            f"{total:.2f}s",
            xy=(x[i], total),
            xytext=(0, 4),
            textcoords="offset points",
            ha="center",
            fontsize=8,
        )

    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylabel("Mean end-to-end latency (s)")
    ax.set_xlabel("Payload size")
    ax.grid(axis="y", linestyle="--", linewidth=0.5, alpha=0.55)
    ax.set_axisbelow(True)

    ax.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, 1.18),
        ncol=4,
        frameon=True,
        edgecolor="black",
        fancybox=False,
    )

    fig.tight_layout()

    pdf_path = FIGURES_DIR / "integration_overhead_stacked_q1.pdf"
    png_path = FIGURES_DIR / "integration_overhead_stacked_q1.png"

    fig.savefig(pdf_path, bbox_inches="tight")
    fig.savefig(png_path, dpi=600, bbox_inches="tight")

    plt.close(fig)

    print(pdf_path)
    print(png_path)


def plot_layer_breakdown_percent(summary_df: pd.DataFrame):
    labels = summary_df["payload_label"].tolist()

    acquisition_s = summary_df["acquisition_ms_mean"].to_numpy() / 1000
    crypto_s = summary_df["crypto_ms_mean"].to_numpy() / 1000
    ipfs_s = summary_df["ipfs_s_mean"].to_numpy()
    blockchain_s = summary_df["blockchain_s_mean"].to_numpy()

    total = acquisition_s + crypto_s + ipfs_s + blockchain_s

    acquisition_pct = acquisition_s / total * 100
    crypto_pct = crypto_s / total * 100
    ipfs_pct = ipfs_s / total * 100
    blockchain_pct = blockchain_s / total * 100

    x = np.arange(len(labels))

    fig, ax = plt.subplots(figsize=(7.2, 4.0))

    ax.bar(x, acquisition_pct, label="Acquisition", color=COLORS["Acquisition"], edgecolor="black", linewidth=0.4)
    ax.bar(x, crypto_pct, bottom=acquisition_pct, label="Crypto", color=COLORS["Crypto"], edgecolor="black", linewidth=0.4)
    ax.bar(x, ipfs_pct, bottom=acquisition_pct + crypto_pct, label="IPFS", color=COLORS["IPFS"], edgecolor="black", linewidth=0.4)
    ax.bar(
        x,
        blockchain_pct,
        bottom=acquisition_pct + crypto_pct + ipfs_pct,
        label="Blockchain",
        color=COLORS["Blockchain"],
        edgecolor="black",
        linewidth=0.4,
    )

    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylabel("Latency contribution (%)")
    ax.set_xlabel("Payload size")
    ax.set_ylim(0, 100)
    ax.grid(axis="y", linestyle="--", linewidth=0.5, alpha=0.55)
    ax.set_axisbelow(True)

    ax.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, 1.18),
        ncol=4,
        frameon=True,
        edgecolor="black",
        fancybox=False,
    )

    fig.tight_layout()

    pdf_path = FIGURES_DIR / "integration_layer_breakdown_q1.pdf"
    png_path = FIGURES_DIR / "integration_layer_breakdown_q1.png"

    fig.savefig(pdf_path, bbox_inches="tight")
    fig.savefig(png_path, dpi=600, bbox_inches="tight")

    plt.close(fig)

    print(pdf_path)
    print(png_path)


# ============================================================
# 6. Main
# ============================================================

if __name__ == "__main__":
    raw_df, summary_df = run_benchmark()

    if not summary_df.empty:
        plot_stacked_latency(summary_df)
        plot_layer_breakdown_percent(summary_df)

        print("\nSummary:")
        print(summary_df.to_string(index=False))
    else:
        print("No successful runs to summarize.")

[OK] 1 KB run=1 | crypto=2.22 ms | IPFS=5.25 s | chain=2.77 s | total=8.04 s
[OK] 1 KB run=2 | crypto=0.36 ms | IPFS=2.24 s | chain=2.69 s | total=4.94 s
[OK] 1 KB run=3 | crypto=0.21 ms | IPFS=2.57 s | chain=2.43 s | total=5.02 s
[OK] 1 KB run=4 | crypto=0.28 ms | IPFS=3.00 s | chain=2.52 s | total=5.55 s
[OK] 1 KB run=5 | crypto=0.23 ms | IPFS=4.39 s | chain=2.82 s | total=7.23 s
[OK] 100 KB run=1 | crypto=4.23 ms | IPFS=5.26 s | chain=3.16 s | total=8.44 s
[OK] 100 KB run=2 | crypto=4.98 ms | IPFS=5.38 s | chain=2.43 s | total=7.84 s
[OK] 100 KB run=3 | crypto=2.25 ms | IPFS=4.03 s | chain=2.59 s | total=6.65 s
[OK] 100 KB run=4 | crypto=1.10 ms | IPFS=3.65 s | chain=2.57 s | total=6.24 s
[OK] 100 KB run=5 | crypto=1.31 ms | IPFS=5.34 s | chain=2.48 s | total=7.84 s
[OK] 1 MB run=1 | crypto=12.04 ms | IPFS=7.98 s | chain=2.74 s | total=10.75 s
[OK] 1 MB run=2 | crypto=11.17 ms | IPFS=9.66 s | chain=2.57 s | total=12.26 s
[OK] 1 MB run=3 | crypto=32.22 ms | IPFS=10.52 s | chain=2.63 

In [1]:
import os
import time
import json
import pandas as pd
from web3 import Web3
from dotenv import load_dotenv

# ============================================================
# 1. Load environment variables
# ============================================================

load_dotenv()

PRIVATE_KEY = os.getenv("PRIVATE_KEY")
IOTA_RPC_URL = os.getenv("IOTA_RPC_URL", "https://json-rpc.evm.testnet.iota.cafe")
CONTRACT_ADDRESS = os.getenv("IOTA_CONTRACT_ADDRESS")

IOTA_USD_RATE = 0.06143  # 1 IOTA = 0.06143 USD

if not PRIVATE_KEY:
    raise ValueError("PRIVATE_KEY is missing in .env")

if not CONTRACT_ADDRESS:
    raise ValueError("IOTA_CONTRACT_ADDRESS is missing in .env")

# ============================================================
# 2. Connect to IOTA EVM
# ============================================================

w3 = Web3(Web3.HTTPProvider(IOTA_RPC_URL, request_kwargs={"timeout": 120}))

if not w3.is_connected():
    raise RuntimeError("Cannot connect to IOTA EVM RPC")

account = w3.eth.account.from_key(PRIVATE_KEY)
sender = account.address

print("=" * 70)
print("Connected to IOTA EVM")
print("Sender:", sender)
print("Contract:", CONTRACT_ADDRESS)
print("=" * 70)

# ============================================================
# 3. Contract ABI
# IMPORTANT:
# This ABI assumes your Solidity function is:
# recordMileage(string vehicleId, string timestamp, string proofCid, uint256 odometerKm)
# If your function name or parameters are different, send me the contract and I adapt it.
# ============================================================

CONTRACT_ABI = [
    {
        "inputs": [
            {"internalType": "string", "name": "vehicleId", "type": "string"},
            {"internalType": "string", "name": "timestamp", "type": "string"},
            {"internalType": "string", "name": "proofCid", "type": "string"},
            {"internalType": "uint256", "name": "odometerKm", "type": "uint256"}
        ],
        "name": "recordMileage",
        "outputs": [],
        "stateMutability": "nonpayable",
        "type": "function"
    },
    {
        "inputs": [
            {"internalType": "string", "name": "vehicleId", "type": "string"}
        ],
        "name": "getVehicle",
        "outputs": [
            {"internalType": "uint256", "name": "lastValidOdometer", "type": "uint256"},
            {"internalType": "bool", "name": "fraudFlag", "type": "bool"},
            {"internalType": "uint256", "name": "totalRecords", "type": "uint256"}
        ],
        "stateMutability": "view",
        "type": "function"
    }
]

contract = w3.eth.contract(
    address=Web3.to_checksum_address(CONTRACT_ADDRESS),
    abi=CONTRACT_ABI
)

# ============================================================
# 4. Workload
# ============================================================

vehicle_id = "V001"

transactions = [
    {"tx_no": 1, "phase": "normal_update", "odometer": 1000},
    {"tx_no": 2, "phase": "normal_update", "odometer": 1100},
    {"tx_no": 3, "phase": "normal_update", "odometer": 1200},
    {"tx_no": 4, "phase": "rollback_attempt", "odometer": 900},
    {"tx_no": 5, "phase": "post_fraud_valid_update", "odometer": 1300},
    {"tx_no": 6, "phase": "post_fraud_valid_update", "odometer": 1400},
    {"tx_no": 7, "phase": "post_fraud_valid_update", "odometer": 1500},
]

# ============================================================
# 5. Helper functions
# ============================================================

def get_dynamic_gas_price():
    """
    Returns current RPC gas price with +20% safety margin.
    """
    network_gas_price = w3.eth.gas_price
    dynamic_gas_price = int(network_gas_price * 1.20)
    return dynamic_gas_price


def get_vehicle_state(vehicle_id):
    """
    Reads vehicle state from the smart contract.
    If your contract does not have getVehicle(), comment this function call.
    """
    try:
        state = contract.functions.getVehicle(vehicle_id).call()
        return {
            "last_valid_odometer": int(state[0]),
            "fraud_flag": bool(state[1]),
            "total_records": int(state[2])
        }
    except Exception as e:
        return {
            "last_valid_odometer": None,
            "fraud_flag": None,
            "total_records": None,
            "state_error": str(e)
        }


def send_record_mileage(tx_no, phase, vehicle_id, odometer):
    """
    Sends one real blockchain transaction.
    """

    timestamp = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
    proof_cid = f"QmSimulatedCID_{vehicle_id}_{tx_no}_{odometer}"

    nonce = w3.eth.get_transaction_count(sender, "pending")
    gas_price = get_dynamic_gas_price()

    gas_price_gwei = float(w3.from_wei(gas_price, "gwei"))

    function_call = contract.functions.recordMileage(
        vehicle_id,
        timestamp,
        proof_cid,
        int(odometer)
    )

    # Estimate gas with safety margin
    try:
        estimated_gas = function_call.estimate_gas({"from": sender})
        gas_limit = int(estimated_gas * 1.25)
    except Exception:
        estimated_gas = None
        gas_limit = 500000

    tx = function_call.build_transaction({
        "from": sender,
        "nonce": nonce,
        "gas": gas_limit,
        "gasPrice": gas_price,
        "chainId": 1076
    })

    signed_tx = account.sign_transaction(tx)

    raw_tx = getattr(signed_tx, "raw_transaction", None)
    if raw_tx is None:
        raw_tx = getattr(signed_tx, "rawTransaction")

    print("\n" + "-" * 70)
    print(f"Tx #{tx_no} | Phase: {phase}")
    print(f"Vehicle ID : {vehicle_id}")
    print(f"Odometer   : {odometer} km")
    print(f"Gas price  : {gas_price_gwei:.4f} gwei")
    print("Sending transaction...")

    start = time.perf_counter()

    tx_hash = w3.eth.send_raw_transaction(raw_tx)
    receipt = w3.eth.wait_for_transaction_receipt(tx_hash, timeout=300)

    end = time.perf_counter()

    latency = end - start
    gas_used = int(receipt["gasUsed"])
    status = int(receipt["status"])

    cost_wei = gas_used * gas_price
    cost_iota = float(w3.from_wei(cost_wei, "ether"))
    cost_usd = cost_iota * IOTA_USD_RATE

    vehicle_state = get_vehicle_state(vehicle_id)

    print("Tx hash    :", tx_hash.hex())
    print("Status     :", status)
    print("Gas used   :", gas_used)
    print(f"Latency    : {latency:.3f} s")
    print(f"Cost IOTA  : {cost_iota:.8f}")
    print(f"Cost USD   : {cost_usd:.8f}")

    if vehicle_state["last_valid_odometer"] is not None:
        print("Last valid :", vehicle_state["last_valid_odometer"])
        print("Fraud flag :", vehicle_state["fraud_flag"])
        print("Records    :", vehicle_state["total_records"])

    return {
        "tx_no": tx_no,
        "phase": phase,
        "vehicle_id": vehicle_id,
        "odometer_km": odometer,
        "tx_hash": tx_hash.hex(),
        "status": status,
        "success": status == 1,
        "gas_price_gwei": gas_price_gwei,
        "gas_used": gas_used,
        "latency_s": latency,
        "cost_iota": cost_iota,
        "cost_usd": cost_usd,
        "last_valid_odometer": vehicle_state.get("last_valid_odometer"),
        "fraud_flag": vehicle_state.get("fraud_flag"),
        "total_records": vehicle_state.get("total_records"),
    }


# ============================================================
# 6. Run benchmark
# ============================================================

results = []

for tx in transactions:
    try:
        result = send_record_mileage(
            tx_no=tx["tx_no"],
            phase=tx["phase"],
            vehicle_id=vehicle_id,
            odometer=tx["odometer"]
        )
        results.append(result)

        # Avoid nonce/RPC pressure
        time.sleep(2)

    except Exception as e:
        print("\nFAILED")
        print("Tx #:", tx["tx_no"])
        print("Error:", str(e))

        results.append({
            "tx_no": tx["tx_no"],
            "phase": tx["phase"],
            "vehicle_id": vehicle_id,
            "odometer_km": tx["odometer"],
            "success": False,
            "error": str(e)
        })

        time.sleep(3)

# ============================================================
# 7. Save results
# ============================================================

df = pd.DataFrame(results)

csv_path = "iota_dynamic_gas_rollback_results.csv"
df.to_csv(csv_path, index=False)

print("\n" + "=" * 70)
print("Results saved to:", csv_path)
print("=" * 70)

print("\nSummary table:")
summary_cols = [
    "tx_no",
    "odometer_km",
    "gas_used",
    "latency_s",
    "fraud_flag",
    "gas_price_gwei",
    "cost_iota",
    "cost_usd"
]

print(df[summary_cols].to_string(index=False))

# ============================================================
# 8. Generate LaTeX rows
# ============================================================

print("\nLaTeX rows:")
for _, row in df.iterrows():
    if row.get("success") is not True:
        continue

    fraud_value = "True" if row["fraud_flag"] else "False"

    print(
        f"{int(row['tx_no'])} & "
        f"{int(row['odometer_km'])} & "
        f"{int(row['gas_used']):,}".replace(",", "{,}") + " & "
        f"{row['latency_s']:.2f} & "
        f"{fraud_value} & "
        f"{row['cost_iota']:.6f} & "
        f"{row['cost_usd']:.6f} \\\\"
    )

Connected to IOTA EVM
Sender: 0xC40f4633b305b90cC0A048dfBF16BAf75Eb1b0C8
Contract: 0x1cede1169D292B18a331728a19d6E4f248249f89

----------------------------------------------------------------------
Tx #1 | Phase: normal_update
Vehicle ID : V001
Odometer   : 1000 km
Gas price  : 12.0000 gwei
Sending transaction...
Tx hash    : 0xc4267a28adaaf3f0264d397d081159dfc52c0d1ab0bb06c781245b3dd87ee187
Status     : 1
Gas used   : 140575
Latency    : 1.891 s
Cost IOTA  : 0.00168690
Cost USD   : 0.00010363

----------------------------------------------------------------------
Tx #2 | Phase: normal_update
Vehicle ID : V001
Odometer   : 1100 km
Gas price  : 12.0000 gwei
Sending transaction...
Tx hash    : 0x7ca0652a65d7aac67b07ce4d24512468faa72bdfb1fc9a2f66d4830e3caf666f
Status     : 1
Gas used   : 123497
Latency    : 1.989 s
Cost IOTA  : 0.00148196
Cost USD   : 0.00009104

----------------------------------------------------------------------
Tx #3 | Phase: normal_update
Vehicle ID : V001
Odometer 